# Titanic Survival Prediction
End-to-end project: data exploration, cleaning, feature engineering, model training, and evaluation.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

sns.set_style("whitegrid")
%matplotlib inline


## 2. Load data

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.isnull().sum()


## 3. Exploratory Data Analysis (EDA)
Let's visualize survival patterns before building the model.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

sns.countplot(data=df, x="Survived", hue="Survived", legend=False, ax=axes[0,0], palette=["#e74c3c", "#2ecc71"])
axes[0,0].set_title("Overall Survival Count")
axes[0,0].set_xticks([0,1]); axes[0,0].set_xticklabels(["Died", "Survived"])

sns.barplot(data=df, x="Sex", y="Survived", hue="Sex", legend=False, ax=axes[0,1], palette="Set2")
axes[0,1].set_title("Survival Rate by Sex"); axes[0,1].set_ylabel("Survival Rate")

sns.barplot(data=df, x="Pclass", y="Survived", hue="Pclass", legend=False, ax=axes[0,2], palette="Set3")
axes[0,2].set_title("Survival Rate by Passenger Class"); axes[0,2].set_ylabel("Survival Rate")

sns.histplot(data=df, x="Age", hue="Survived", multiple="stack", bins=30, ax=axes[1,0], palette=["#e74c3c", "#2ecc71"])
axes[1,0].set_title("Age Distribution by Survival")

sns.boxplot(data=df, x="Survived", y="Fare", hue="Survived", legend=False, ax=axes[1,1], palette=["#e74c3c", "#2ecc71"])
axes[1,1].set_title("Fare Distribution by Survival")
axes[1,1].set_xticks([0,1]); axes[1,1].set_xticklabels(["Died", "Survived"]); axes[1,1].set_ylim(0, 300)

sns.barplot(data=df, x="Embarked", y="Survived", hue="Embarked", legend=False, ax=axes[1,2], palette="Set1")
axes[1,2].set_title("Survival Rate by Embarkation Port"); axes[1,2].set_ylabel("Survival Rate")

plt.tight_layout()
plt.show()


**Key observations:**
- Overall, more passengers died than survived.
- Females had a much higher survival rate than males.
- 1st class passengers survived at a much higher rate than 3rd class.
- Children and younger passengers had somewhat better survival odds.
- Higher fare (linked to class) correlates with survival.
- Passengers who embarked at Cherbourg (C) had a higher survival rate — likely because more 1st class passengers boarded there.


In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df[["Survived","Pclass","Age","SibSp","Parch","Fare"]].corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (numeric features)")
plt.show()


## 4. Feature Engineering

In [ ]:
# Title extracted from Name (Mr, Mrs, Miss, Master, Rare, etc.)
df["Title"] = df["Name"].str.extract(r",\s*([^\.]*)\.")
rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
df["Title"] = df["Title"].replace(rare_titles, "Rare")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

# Family size features
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Deck from Cabin (first letter); missing cabin -> 'U' (unknown)
df["Deck"] = df["Cabin"].str[0].fillna("U")

# Fill missing Age using median age per Title
df["Age"] = df.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill missing Embarked with mode, Fare with median
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Age & Fare bins
df["AgeBin"] = pd.cut(df["Age"], bins=[0, 12, 20, 40, 60, 100],
                       labels=["Child", "Teen", "Adult", "MiddleAge", "Senior"])
df["FareBin"] = pd.qcut(df["Fare"], 4, labels=["Low", "Mid", "High", "VeryHigh"])

df[["Title","FamilySize","IsAlone","Deck","AgeBin","FareBin"]].head()


In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=df, x="Title", y="Survived", hue="Title", legend=False, palette="viridis")
plt.title("Survival Rate by Title")
plt.ylabel("Survival Rate")
plt.show()


## 5. Prepare features for modeling

In [ ]:
features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title",
            "FamilySize", "IsAlone", "Deck", "AgeBin", "FareBin"]

X = df[features].copy()
y = df["Survived"]

X = pd.get_dummies(X, columns=["Sex", "Embarked", "Title", "Deck", "AgeBin", "FareBin"], drop_first=True)
X.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


## 6. Train models

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_preds = log_reg.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
print("Models trained.")


## 7. Evaluate models

In [ ]:
def evaluate(name, y_true, y_pred, y_proba=None):
    print(f"===== {name} =====")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    if y_proba is not None:
        print("ROC AUC :", round(roc_auc_score(y_true, y_proba), 4))
    print(classification_report(y_true, y_pred, target_names=["Died", "Survived"]))

evaluate("Logistic Regression", y_test, log_preds)
evaluate("Random Forest", y_test, rf_preds, rf_proba)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.heatmap(confusion_matrix(y_test, log_preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Died","Survived"], yticklabels=["Died","Survived"], ax=axes[0])
axes[0].set_title("Logistic Regression - Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

sns.heatmap(confusion_matrix(y_test, rf_preds), annot=True, fmt="d", cmap="Greens",
            xticklabels=["Died","Survived"], yticklabels=["Died","Survived"], ax=axes[1])
axes[1].set_title("Random Forest - Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
cv_scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print("Random Forest 5-fold CV accuracy: {:.4f} (+/- {:.4f})".format(cv_scores.mean(), cv_scores.std()))


## 8. Feature importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False, palette="mako")
plt.title("Top 10 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.show()

importances


## 9. Summary

- **Logistic Regression accuracy:** ~84%
- **Random Forest accuracy:** ~83% (5-fold CV: ~83.6%)
- **Most important predictors:** Sex, Title (Mr/Mrs/Miss), Fare, Pclass, Age
- This confirms the historical "women and children first" pattern, along with a strong class/wealth effect on survival.
